# Electrical Resistivity Tomography (ERT)

**Near-Surface Geophysics — Missouri S&T**

A resistivity survey pushes current into the ground through two electrodes and
measures the voltage it makes between two others. Repeat that for hundreds of
four-electrode combinations along a line of electrodes and you have data that
see to different depths at different places. **ERT** turns those numbers into
a 2-D picture of the ground's resistivity.

This notebook works through one AGI SuperSting dipole-dipole line, exported in
the RES2DINV `.dat` format, with the `shallowgeo.ert` module (NumPy and SciPy
only).

1. **Read** the file and repair what is wrong with a SuperSting export:
   non-integer *n*, and readings made with electrodes that were never in the
   ground.
2. **The forward problem**: geometric factors, apparent resistivity, and
   what a pseudosection does and does not show.
3. **Invert synthetic data** recorded with this line's exact measurements, to
   see what the survey can resolve, how to choose the smoothing, and how deep
   it sees.
4. **Check the field data before believing them**, with tests that need no
   inversion.
5. **Invert the field data** and read the misfit.
6. **Your turn**: change the inversion and regularization settings and see
   what changes in the picture, and what does not.

Units are **metres** and **ohm-metres** (Ω·m) throughout.

> **The one thing to take away.** An inversion always returns a picture. Whether
> the picture means anything is decided by the misfit and by checks on the data
> that need no inversion at all.

Questions marked **Q** are for your write-up.

## 0. Setup

Run this cell first. It installs `shallowgeo` if it is missing (or out of
date) and imports what the notebook uses.

In [ ]:
# Checks for the *ERT* code specifically: an older install imports fine and
# then lacks it, and since the version string does not change between builds,
# a plain `pip install` reports "already satisfied".
import importlib
import sys

PACKAGE = ("shallow-geophysics @ "
           "git+https://github.com/Maurer-GEMLab/shallow-geophysics.git")

def shallowgeo_ready():
    for name in [n for n in sys.modules
                 if n == "shallowgeo" or n.startswith("shallowgeo.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    try:
        from shallowgeo.ert import resistivity_tomography  # noqa: F401
        from shallowgeo.datasets import example_data  # noqa: F401
        return True
    except ImportError:
        return False

if not shallowgeo_ready():
    %pip install -q --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    %pip install -q --force-reinstall --no-deps --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    raise ImportError(
        "shallowgeo.ert is still missing after installing. "
        "In Colab: Runtime > Restart session, then run this cell again."
    )

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shallowgeo.ert import (
    read_res2dinv, dipole_dipole, make_mesh, forward, apparent_resistivity,
    resistivity_tomography, lambda_sweep, doi_index,
)
from shallowgeo.datasets import example_data

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
print("ready")

## 1. The data

Set `DATA` to your `.dat` file and `N_ELECTRODES` to the number of electrodes
that were **in the ground** (from the field notes, not from the file). In
Colab, if the file is not already in the session, the cell asks you to upload
it.

In [ ]:
DATA = Path("1109261.dat")   # your RES2DINV .dat file
N_ELECTRODES = 42            # electrodes actually in the ground

# Two lines that ship with the package: one that passes these checks and
# one that fails them.
# DATA, N_ELECTRODES = example_data("2026-11-11-ert-line") / "111126.dat", 21
# DATA, N_ELECTRODES = example_data("2610234-ert-line") / "2610234.dat", 42

if not Path(DATA).exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            f"{DATA} not found: put the .dat file next to this notebook "
            "or set DATA to its full path") from None
    print(f"{DATA} is not in this Colab session: choose it in the upload dialog")
    DATA = Path(next(iter(files.upload())))

print(*Path(DATA).read_text().splitlines()[:16], sep="\n")

The six header lines are the title, the smallest electrode spacing, the array
type (`3` = dipole-dipole), the number of measurements, the x-location
convention (`1` = the array's mid-point) and an IP flag. Each data line is
**x, a, n, ρa**: where the array is, its dipole length, the separation between
the dipoles in multiples of *a*, and the apparent resistivity.

Dipole-dipole puts the current pair (C2 C1) on one side and the potential pair
(P1 P2) on the other:

```
  C2 ─a─ C1 ─────── n·a ─────── P1 ─a─ P2
```

Scroll down the raw file and you may find `n` written as `9.99999` or
`8.00039`. The SuperSting computes these from electrode positions held in
single precision, and **RES2DINV refuses a non-integer `n`.**

`read_res2dinv` does not trust the `x a n` columns. It works out the four
electrode positions of every measurement, snaps each to the nearest multiple
of the spacing, and numbers the electrodes. That repairs `a`, `n` and `x`
together, and reports how far it had to move anything.

One more thing about this file: the SuperSting's own record (the `.stg`
file) also holds readings with **negative** apparent resistivity, and the
conversion to `.dat` leaves them out. `read_stg` reads the `.stg` directly if
you have it.

In [ ]:
raw = read_res2dinv(DATA)
print(raw)
print(f"largest snap: {1000 * raw.metadata['snapped_max_m']:.2f} mm")
raw.data.head()

Each row now has the electrode numbers `a b m n` (current pair, potential pair,
counting from 0), the apparent resistivity `rhoa`, the geometric factor `k`,
the transfer resistance `r` = ρa / k = V/I, and the position of the
measurement on a pseudosection (`x`, `depth`).

### Readings past the end of the cable

A SuperSting command file is written for a number of electrodes. If fewer
were laid out, the instrument still makes every reading the command file asks
for, and readings that use an address with **no electrode on it** go into the
file like any other. They are not data.

The left plot shows each reading's apparent resistivity against the highest
electrode number it uses; the right plot shows where those readings sit on
the pseudosection.

In [ ]:
idx = raw.data[list("abmn")].to_numpy()
furthest = idx.max(axis=1) + 1          # electrode number, counting from 1
past = furthest > N_ELECTRODES
print(f"the file uses {raw.n_electrodes} electrode positions; "
      f"{past.sum()} of {len(raw)} readings use an electrode past number {N_ELECTRODES}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.semilogy(furthest[~past], raw.data["rhoa"][~past], ".", ms=5, label="on the cable")
ax.semilogy(furthest[past], raw.data["rhoa"][past], ".", ms=5, color="#d1495b",
            label=f"uses an electrode past number {N_ELECTRODES}")
ax.axvline(N_ELECTRODES + 0.5, color="k", ls="--", lw=1)
ax.set_xlabel("highest electrode number the reading uses")
ax.set_ylabel("apparent resistivity (Ω·m)")
ax.legend(fontsize=8)
ax = axes[1]
ax.scatter(raw.data["x"][~past], raw.data["depth"][~past], s=10, label="on the cable")
ax.scatter(raw.data["x"][past], raw.data["depth"][past], s=10, color="#d1495b",
           label="uses an electrode that is not there")
ax.axvline(raw.electrodes[min(N_ELECTRODES, raw.n_electrodes) - 1, 0], color="k", ls="--", lw=1)
ax.set_ylim(raw.data["depth"].max() * 1.05, 0)
ax.set_xlabel("measurement mid-point (m)")
ax.set_ylabel("pseudo-depth (m)")
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

**Q1.** Readings made on electrodes in the ground should form a band of
apparent resistivities that changes gradually. Does the character of the data
change at the dashed line? If the change is somewhere else, the field notes
and the data disagree: find out which is right before going on (try other
values of `N_ELECTRODES`).

A reading can have its mid-point well inside the line and still reach past
the end: a long dipole-dipole spans (n + 2)·a. `trim` drops every measurement
that uses any electrode beyond the last one in the ground, and records what
it did in the data's provenance. `to_res2dinv` writes a file RES2DINV will
load: integer `n`, and a header count that matches. From a terminal the same
repair is one command: `shallowgeo ert-clean <file>.dat -e <N_ELECTRODES>`.

In [ ]:
data = raw.trim(N_ELECTRODES)
print(data)
print(data.provenance[-1])
clean_file = data.to_res2dinv(f"{Path(DATA).stem}_clean.dat")
print(*clean_file.read_text().splitlines()[:9], sep="\n")

### Readings missing on the cable

Trimming looks in one direction. The other is to ask what the file *should*
hold on these electrodes. Every kept reading has a configuration (a dipole
length and an *n*), and each configuration fits at a known number of
positions along the cable. `coverage()` counts how many of those positions the
file has, and `missing_readings()` lists the rest. `ert-clean` prints the same
counts.

In [ ]:
cov = data.coverage()
print(cov.drop(columns="offsets").to_string(index=False))
print(f"\n{cov['measured'].sum()} of the {cov['possible'].sum()} readings these "
      f"configurations allow on {N_ELECTRODES} electrodes are in the file")

missing = data.missing_readings()
fig, ax = plt.subplots(figsize=(10, 3.5))
if len(missing):
    ax.scatter(missing["x"], missing["depth"], s=16, facecolors="none", edgecolors="0.6",
               label=f"allowed by the scheme, not in the file ({len(missing)})")
ax.scatter(data.data["x"], data.data["depth"], s=12, label=f"in the file ({len(data)})")
ax.plot(data.electrodes[:, 0], np.zeros(data.n_electrodes), "kv", ms=3, clip_on=False)
deepest = max([data.data["depth"].max()] + ([missing["depth"].max()] if len(missing) else []))
ax.set_ylim(deepest * 1.05, 0)
ax.set_xlabel("measurement mid-point (m)")
ax.set_ylabel("pseudo-depth (m)")
ax.legend(loc="lower right", fontsize=8)
ax.set_title("readings on the cable: what the file has, and what it lacks", fontsize=10)
plt.show()

**Q2.** Is anything missing? Gaps found here were in the file before
trimming (`trim` only removes readings that use an electrode past the last
one). They come from skipped injections, a run cut short, or readings the
`.dat` export left out — which, for this instrument, includes every negative
reading. Wherever the pseudosection has holes, every picture made from it
rests on fewer readings.

Which dipole lengths and which *n* does this command file use? Where along
the line are the deepest readings?

## 2. The forward problem

### Apparent resistivity

Current *I* injected at one electrode on uniform ground of resistivity ρ
raises the potential at distance *r* to V = ρI / (2πr). With two current and
two potential electrodes the voltage measured is

$$ V = \frac{\rho I}{2\pi}\left(\frac{1}{AM} - \frac{1}{BM} - \frac{1}{AN} + \frac{1}{BN}\right)
   \quad\Rightarrow\quad \rho_a = k\,\frac{V}{I}, \qquad
   k = \frac{2\pi}{1/AM - 1/BM - 1/AN + 1/BN}. $$

For dipole-dipole, k = π a n(n+1)(n+2): it grows as n³, because the voltage
falls as n⁻³. Over uniform ground ρa is ρ for every measurement. Over real
ground it is a weighted average of everything the current flows through — an
*apparent* resistivity.

### The pseudosection

Plotting each ρa at its mid-point and at its **median depth of investigation**
(Edwards, 1977; about 0.42 a for n = 1, 1.7 a for n = 6) gives a
*pseudosection*. It is a picture of the data, not of the ground.

### Modelling

`forward` solves for the potential on a finite-element mesh. The ground is
2-D but the source is a point, so the potential is 3-D; a Fourier transform
across the line turns it into a set of 2-D problems, one per wavenumber
(the "2.5-D" method). The mesh is a quarter of an electrode spacing wide under
the electrodes and grows geometrically outwards. Its error against the exact
solution for two layers is under 1 %.

Here is what a buried conductor does to a dipole-dipole pseudosection,
recorded with **this line's own measurements**. The model is scaled to the
length of the line: 80 Ω·m soil over 400 Ω·m, with a 20 Ω·m body under the
middle.

In [ ]:
mesh = make_mesh(data)
print(mesh)
L = np.ptp(data.electrodes[:, 0])     # line length (m)
SOIL = 0.04 * L                       # soil thickness (m)

def model(X, Z):
    """80 ohm-m soil over 400 ohm-m, with a 20 ohm-m body under the middle."""
    rho = np.where(Z < SOIL, 80.0, 400.0)
    body = (np.abs(X - data.electrodes[0, 0] - L / 2) < 0.065 * L) & (Z > SOIL) & (Z < 0.105 * L)
    return np.where(body, 20.0, rho)

X, Z = mesh.mesh()
true = model(X, Z)
synthetic = apparent_resistivity(data, model, mesh=mesh, noise=0.03, seed=1)

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5))
im = axes[0].pcolormesh(mesh.x_edges, mesh.z_edges, true, cmap="turbo",
                        norm=plt.matplotlib.colors.LogNorm(20, 400))
fig.colorbar(im, ax=axes[0], label="resistivity (Ω·m)", shrink=0.8, pad=0.02,
             ticks=[20, 50, 100, 200, 400], format="%g")
axes[0].set_ylim(mesh.z_edges[-1], 0); axes[0].set_aspect("equal")
axes[0].set_title("true model", fontsize=10)
axes[0].set_ylabel("depth (m)")
synthetic.plot_pseudosection(axes[1])
axes[1].set_title("its pseudosection (3 % noise), this line's measurement scheme", fontsize=10)
plt.tight_layout(); plt.show()

**Q3.** Where is the conductor in the pseudosection, and what shape does it
take? Why is a pseudosection not a cross-section of the ground?

## 3. Inversion

The model is the logarithm of resistivity in each cell of the model grid.
Each Gauss-Newton step solves

$$ \min_m\; \left\| \frac{\ln\rho_a^{obs} - \ln\rho_a(m)}{e} \right\|^2
   + \lambda \left( \|R\,m\|^2 + \alpha_s \|m - m_{ref}\|^2 \right) $$

where *e* is the relative data error (3 % here, the noise that was added), R
takes differences between neighbouring cells — vertical ones weighted by
`z_weight` — and the optional α<sub>s</sub> term pulls cells the data do not
see towards a reference model. **λ trades fit against smoothness.** The
sensitivities come from the same potentials as the forward solve (by
reciprocity), so each step costs about two forward solves.

The misfit is reported two ways: **RMS**, the root-mean-square difference
between measured and calculated ρa in percent, and **χ²**, the mean squared
misfit divided by the assumed error. χ² ≈ 1 means the model fits the data as
well as the noise allows.

In [ ]:
res = resistivity_tomography(synthetic, mesh=mesh, lam=20, verbose=True)
print(res.summary())

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5))
res.plot(axes[0], vmin=20, vmax=400)
axes[0].set_title("recovered from the synthetic data", fontsize=10)
res.plot(axes[1], field="coverage")
axes[1].set_title("relative coverage (error-weighted cumulative sensitivity)", fontsize=10)
plt.tight_layout(); plt.show()

The image is **faded where coverage falls below 1 % of its maximum** — those
colours are the smoothness constraint filling in, not the ground.

**Q4.** Compare the recovered image with the true model. Is the layer boundary
sharp? Is the body the right size, the right resistivity, at the right depth?
Is its base resolved? (A smoothness-constrained inversion spreads things out;
that is what it is designed to do, not a bug.)

### Choosing λ

Too small and the image grows artefacts to fit the noise; too large and it
cannot fit the data. The usual choice is the smoothest model that fits the
data to the noise level, χ² ≈ 1.

In [ ]:
sweep = lambda_sweep(synthetic, lams=(2, 6, 20, 60, 200), mesh=mesh)
print(sweep[["lam", "chi2", "rms_pct", "roughness"]].round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
axes[0].loglog(sweep["roughness"], sweep["chi2"], "o-")
for _, r in sweep.iterrows():
    axes[0].annotate(f"λ={r['lam']:g}", (r["roughness"], r["chi2"]), fontsize=8)
axes[0].axhline(1, color="k", ls="--", lw=0.8)
axes[0].set_xlabel("model roughness |Rm|"); axes[0].set_ylabel("chi²")
for ax, i in zip(axes[1:], (0, len(sweep) - 1)):
    sweep["result"].iloc[i].plot(ax, vmin=20, vmax=400, colorbar=False)
    ax.set_title(f"λ = {sweep['lam'].iloc[i]:g}", fontsize=10)
plt.tight_layout(); plt.show()

**Q5.** Which λ would you pick, and why? What goes wrong in the image at the
smallest λ, and at the largest?

### How deep does it see?

Coverage is one guide. A stricter one is the **depth-of-investigation index**
(Oldenburg & Li, 1999): invert twice, starting from and pulled weakly towards
reference models ten times above and ten times below the median apparent
resistivity. Where the data control the model both runs agree (index near 0);
where they do not, each stays near its own reference (index near 1).

In [ ]:
doi, low, high = doi_index(synthetic, mesh=mesh, lam=20)

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5))
im = axes[0].pcolormesh(mesh.x_edges, mesh.z_edges, doi, cmap="viridis", vmin=0, vmax=1)
axes[0].contour(X, Z, doi, levels=[0.2], colors="w")
fig.colorbar(im, ax=axes[0], label="DOI index", shrink=0.8, pad=0.02)
axes[0].set_ylim(mesh.z_edges[-1], 0); axes[0].set_aspect("equal")
axes[0].set_ylabel("depth (m)")
axes[0].set_title("DOI index (white: 0.2)", fontsize=10)
res.plot(axes[1], mask=doi < 0.2, vmin=20, vmax=400)
axes[1].set_title("the image, faded where DOI > 0.2", fontsize=10)
plt.tight_layout(); plt.show()

## 4. Check the field data before inverting them

Nothing so far needed the field data. Before inverting them, look at them in
ways that need no inversion at all.

### Test 1: how fast can ρa change with *n*?

Below are the dipole-length-1 readings, median apparent resistivity and
transfer resistance against *n*. The dashed curves are the most extreme
layered ground there is: a thin layer with the n = 1 resistivity on a perfect
insulator. Over that, dipole-dipole ρa rises as fast as any layered ground
can make it.

In [ ]:
d = data.data
dip = (d["a"] - d["b"]).abs()
sep = (d["m"] - d["a"]) // dip
d1 = d[dip == 1]
by_n = d1.groupby(sep[dip == 1])[["rhoa", "r"]].median()
by_n.index.name = "n"
print(by_n.round(3))

dd = dipole_dipole(N_ELECTRODES, data.spacing, n_max=int(by_n.index.max()))
mesh_dd = make_mesh(dd)
_, Zdd = mesh_dd.mesh()
dd_sep = (dd.data["m"] - dd.data["a"]).to_numpy()
extreme = {}
for f in (0.3, 1.0, 2.0):
    h = f * data.spacing
    rhoa = forward(mesh_dd, np.where(Zdd < h, by_n["rhoa"].iloc[0], 1e6), dd)["rhoa"]
    extreme[h] = [np.median(rhoa[dd_sep == n]) for n in by_n.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(by_n.index, by_n["rhoa"], "o-", color="#d1495b", label="field data")
for h, v in extreme.items():
    axes[0].semilogy(by_n.index, v, "--", label=f"{h:.2f} m of soil on an insulator")
axes[0].axhline(by_n["rhoa"].iloc[0], color="0.5", lw=0.8, label="uniform ground")
axes[0].set_xlabel("n"); axes[0].set_ylabel("apparent resistivity (Ω·m)")
axes[0].legend(fontsize=8)
n = by_n.index.to_numpy()
k = np.pi * data.spacing * n * (n + 1) * (n + 2)
axes[1].semilogy(n, by_n["r"], "o-", color="#d1495b", label="field data")
axes[1].semilogy(n, by_n["rhoa"].iloc[0] / k, color="0.5", label="uniform ground")
axes[1].set_xlabel("n"); axes[1].set_ylabel("transfer resistance V/I (Ω)")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Q6.** Does the field ρa rise any faster with *n* than the dashed curves
allow? (A fall with *n* is possible: it means more conductive ground at
depth.) Does the transfer resistance fall with *n* roughly like the
uniform-ground curve? What would it mean if the voltage barely fell as the
potential electrodes moved away from the current?

### Test 2: readings from electrodes that were not there

`trim` removed the readings that used electrodes past the cable. A potential
electrode attached to nothing cannot measure the ground. Compare those readings
with the ones made on the cable:

In [ ]:
inside = ~past
cur_on_cable = (idx[:, :2] < N_ELECTRODES).all(axis=1)
print(f"removed readings with both current electrodes on the cable: "
      f"{(past & cur_on_cable).sum()} of {past.sum()}")

if past.sum() == 0:
    print("no readings past the cable in this file: nothing to compare")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for ax, col, label in ((axes[0], "r", "transfer resistance V/I (Ω)"),
                           (axes[1], "rhoa", "apparent resistivity (Ω·m)")):
        v = raw.data[col]
        bins = np.geomspace(v[v > 0].min(), v.max(), 40)
        ax.hist(v[inside], bins=bins, alpha=0.6, label=f"electrodes in the ground ({inside.sum()})")
        ax.hist(v[past], bins=bins, alpha=0.6, color="#d1495b",
                label=f"electrodes not there ({past.sum()})")
        ax.set_xscale("log"); ax.set_xlabel(label); ax.set_ylabel("readings")
        ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    print(raw.data.groupby(np.where(past, "not there", "in the ground"))[["r", "rhoa"]]
          .describe().T.round(2))

**Q7.** Do the readings from electrodes that were not there look like the
rest? If they are wildly different — scattered, orders of magnitude off — the
instrument is measuring the ground where the electrodes are, and junk where
they are not, which is what should happen. If they look *just like* the rest,
the recorded voltages are dominated by something other than the ground
(leakage in a wet or damaged cable or connector, a cable that does not match
the command file), and trimming cannot make the remaining readings real.

### Test 3: does the reading follow the geometry or the receiver channel?

The SuperSting measures up to eight potential dipoles at once for each current
injection, one per receiver channel. Tabulate the median transfer resistance
by dipole length and *n*:

In [ ]:
d = data.data.copy()
d["dipole"] = (d["a"] - d["b"]).abs()
d["n_sep"] = (d["m"] - d["a"]) // d["dipole"]
table = d.pivot_table(index="n_sep", columns="dipole", values="r", aggfunc="median")
print("median transfer resistance V/I (ohm) by n (rows) and dipole length (columns):")
print(table.round(4))

**Q8.** In each column, V/I should fall steadily with *n*, and different
columns should differ (the geometric factor grows as a·n³). If the same short
series of values repeats in every column, the reading follows its place in
the eight-channel set, not where its electrodes are — ground cannot do that;
an instrument or cable fault can. What does this line show?

Write down your verdict on the data **before** you look at the inversion.

## 5. Invert the field data

The field data have no known noise level. Start by assuming 5 % error and the
same λ as the synthetic test. A few bad readings (poor electrode contact, a
noisy channel) can pull the whole image around, so the first pass is used to
find readings the model cannot fit without bending around them alone —
`outliers()` flags misfits more than three robust standard deviations from
the rest — and the second pass leaves them out.

In [ ]:
first = resistivity_tomography(data, mesh=mesh, lam=20, rel_error=0.05, verbose=True)
print(first.summary())

KEY = list("abmn")
flagged = first.outliers(k=3.0)
is_outlier = data.data[KEY].apply(tuple, axis=1).isin(set(flagged[KEY].apply(tuple, axis=1)))
kept = data.select(~is_outlier.to_numpy(), reason="drop misfit outliers (3 robust SD)")
print(f"\n{is_outlier.sum()} of {len(data)} readings flagged as outliers")

fig, ax = plt.subplots(figsize=(10, 3.5))
data.plot_pseudosection(ax)
ax.scatter(data.data["x"][is_outlier], data.data["depth"][is_outlier], s=60,
           facecolors="none", edgecolors="k", label="outlier")
ax.legend(fontsize=8, loc="lower right")
ax.set_title("measured apparent resistivity, outliers circled", fontsize=10)
plt.show()

In [ ]:
field = resistivity_tomography(kept, mesh=mesh, lam=20, rel_error=0.05, verbose=True)
print(field.summary())

fig = plt.figure(figsize=(10, 12))
gs = fig.add_gridspec(4, 1, height_ratios=[1.4, 1, 1, 1])
ax0 = fig.add_subplot(gs[0])
field.plot(ax0)
ax0.set_title(f"field data: {len(kept)} readings, RMS {field.rms:.1f} %, χ² {field.chi2:.2f}",
              fontsize=10)
field.plot_pseudosections([fig.add_subplot(gs[i]) for i in (1, 2, 3)])
plt.tight_layout(); plt.show()

field.plot_fit()
plt.tight_layout(); plt.show()

**Q9.** How well does the model fit? Compare χ² and RMS with the synthetic
inversion, where the noise was known to be 3 %. Is the remaining misfit random
(speckled red and blue) or systematic (patches, or a trend with depth)? Which
readings fit worst — short or long dipoles, small or large *n*? Does this
agree with your verdict from section 4?

**Q10.** Describe the image: what is resistive, what is conductive, at what
depths? Which features would you trust, and which might be the smoothing or
a handful of readings?

## 6. Your turn: inversion and regularization settings

Change the settings in the next cell and re-run it and the cell after it.
Every run is added to a table at the bottom so you can compare them. Change
**one thing at a time**.

| setting | what it does |
|---|---|
| `LAM` | regularization strength λ: larger is smoother and fits worse, smaller fits better and grows artefacts |
| `Z_WEIGHT` | weight of vertical smoothing relative to horizontal; below 1 lets resistivity change faster with depth than along the line (layers), above 1 favours vertical structure |
| `REL_ERROR` | the error you assume for every reading; χ² is measured against it |
| `ALPHA_S`, `REFERENCE` | the pull towards a reference resistivity, and that resistivity (Ω·m; `None` = median ρa). It only matters where the data say little |
| `MAX_ITER` | the most Gauss-Newton steps |
| `MAX_N`, `MAX_DIPOLE` | leave out readings with larger *n* or longer dipoles (the noisiest ones) |
| `OUTLIER_K` | outlier cut in robust standard deviations (`None` keeps every reading) |
| `MESH_DEPTH`, `CELL_WIDTH` | depth of the model grid and width of its cells (m; `None` = automatic) |
| `COLOR_LIMITS` | colour range of the image in Ω·m, so different runs can be compared |

In [ ]:
LAM = 20
Z_WEIGHT = 0.5
REL_ERROR = 0.05
ALPHA_S = 0.0
REFERENCE = None
MAX_ITER = 20
MAX_N = None
MAX_DIPOLE = None
OUTLIER_K = 3.0
MESH_DEPTH = None
CELL_WIDTH = None
COLOR_LIMITS = (None, None)

In [ ]:
def run_inversion():
    d = data.data
    dip = (d["a"] - d["b"]).abs() // 1
    sep = (d["m"] - d["a"]) // dip
    keep = np.ones(len(d), bool)
    if MAX_N is not None:
        keep &= (sep <= MAX_N).to_numpy()
    if MAX_DIPOLE is not None:
        keep &= (dip <= MAX_DIPOLE).to_numpy()
    sub = data.select(keep, reason="reading selection")
    m = make_mesh(sub, depth=MESH_DEPTH, dx=CELL_WIDTH)
    opts = dict(mesh=m, lam=LAM, z_weight=Z_WEIGHT, rel_error=REL_ERROR, alpha_s=ALPHA_S,
                start=REFERENCE, reference=REFERENCE, max_iter=MAX_ITER)
    result = resistivity_tomography(sub, **opts)
    n_out = 0
    if OUTLIER_K is not None:
        bad = result.outliers(k=OUTLIER_K)
        out = sub.data[KEY].apply(tuple, axis=1).isin(set(bad[KEY].apply(tuple, axis=1)))
        n_out = int(out.sum())
        if n_out:
            sub = sub.select(~out.to_numpy(), reason="drop misfit outliers")
            result = resistivity_tomography(sub, **opts)
    return result, n_out


trial, n_out = run_inversion()

fig = plt.figure(figsize=(10, 7))
gs = fig.add_gridspec(2, 1, height_ratios=[1.4, 1])
ax0 = fig.add_subplot(gs[0])
trial.plot(ax0, vmin=COLOR_LIMITS[0], vmax=COLOR_LIMITS[1])
ax0.set_title(f"λ={LAM:g}, z_weight={Z_WEIGHT:g}, error={100 * REL_ERROR:g} %: "
              f"{len(trial.data)} readings, RMS {trial.rms:.1f} %, χ² {trial.chi2:.2f}",
              fontsize=10)
trial.data.plot_pseudosection(fig.add_subplot(gs[1]), values=trial.data.data["residual"],
                              log=False, cmap="RdBu_r", label="misfit (%)")
plt.tight_layout(); plt.show()

try:
    trials
except NameError:
    trials = []
trials.append({"lam": LAM, "z_weight": Z_WEIGHT, "rel_error": REL_ERROR, "alpha_s": ALPHA_S,
               "reference": REFERENCE, "max_n": MAX_N, "max_dipole": MAX_DIPOLE,
               "outlier_k": OUTLIER_K, "outliers_dropped": n_out,
               "readings": len(trial.data), "iterations": int(trial.history["iteration"].iloc[-1]),
               "chi2": round(trial.chi2, 2), "rms_pct": round(trial.rms, 1),
               "rho_min": round(float(trial.resistivity.min()), 1),
               "rho_max": round(float(trial.resistivity.max()), 1)})
pd.DataFrame(trials)

### The trade-off curve for the field data

The same λ sweep as in section 3, now on the field data, using the
`Z_WEIGHT` and `REL_ERROR` set above. Unlike the synthetic data, the
real noise level is unknown, so χ² = 1 may not be reachable: look for the
corner of the curve, where making the model rougher stops buying much fit.

In [ ]:
LAMS = (3, 10, 30, 100, 300)
sweep_f = lambda_sweep(kept, lams=LAMS, mesh=mesh, z_weight=Z_WEIGHT, rel_error=REL_ERROR)
print(sweep_f[["lam", "chi2", "rms_pct", "roughness"]].round(2))

fig, axes = plt.subplots(len(LAMS) + 1, 1, figsize=(10, 3.2 * (len(LAMS) + 1)))
axes[0].loglog(sweep_f["roughness"], sweep_f["chi2"], "o-")
for _, r in sweep_f.iterrows():
    axes[0].annotate(f"λ={r['lam']:g}", (r["roughness"], r["chi2"]), fontsize=8)
axes[0].axhline(1, color="k", ls="--", lw=0.8)
axes[0].set_xlabel("model roughness |Rm|"); axes[0].set_ylabel("chi²")
lo = min(r.resistivity.min() for r in sweep_f["result"])
hi = max(r.resistivity.max() for r in sweep_f["result"])
for ax, (_, r) in zip(axes[1:], sweep_f.iterrows()):
    r["result"].plot(ax, vmin=lo, vmax=hi)
    ax.set_title(f"λ = {r['lam']:g}: RMS {r['rms_pct']:.1f} %, χ² {r['chi2']:.2f}", fontsize=10)
plt.tight_layout(); plt.show()

### How deep does the field line see?

In [ ]:
doi_f, _, _ = doi_index(kept, mesh=mesh, lam=LAM, z_weight=Z_WEIGHT, rel_error=REL_ERROR)

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5))
im = axes[0].pcolormesh(mesh.x_edges, mesh.z_edges, doi_f, cmap="viridis", vmin=0, vmax=1)
axes[0].contour(X, Z, doi_f, levels=[0.2], colors="w")
fig.colorbar(im, ax=axes[0], label="DOI index", shrink=0.8, pad=0.02)
axes[0].set_ylim(mesh.z_edges[-1], 0); axes[0].set_aspect("equal")
axes[0].set_ylabel("depth (m)")
axes[0].set_title("field data: DOI index (white: 0.2)", fontsize=10)
field.plot(axes[1], mask=doi_f < 0.2)
axes[1].set_title("field image, faded where DOI > 0.2", fontsize=10)
plt.tight_layout(); plt.show()

### Things to try

1. **λ.** Run with `LAM` = 2, 20 and 200. Which features survive every
   value? Those are the ones the data demand.
2. **Assumed error.** Set `REL_ERROR` to 0.02 and then 0.10 with `LAM` fixed.
   How do χ² and the image change? Why does χ² alone not tell you the right
   error?
3. **Smoothing direction.** Try `Z_WEIGHT` = 0.2, 1 and 3. Which geology does
   each favour? Does the misfit say one is better?
4. **Noisy readings.** Set `MAX_N = 8`, then `MAX_DIPOLE = 1`. What do you
   lose at depth, and what happens to the misfit?
5. **Outliers.** Set `OUTLIER_K = None` (keep every reading). Which parts of
   the image move?
6. **Reference model.** Set `ALPHA_S = 0.05` and `REFERENCE` to 10 and then
   1000. Where does the image change? Compare with the DOI plot above.
7. **Mesh.** Halve `CELL_WIDTH` and double `MESH_DEPTH`. Does anything
   important change? How long does it take?

**Q11.** Pick your preferred settings and justify each one. Which features of
your final image are robust to every reasonable choice?

## Summary

- **Repair the file first**: integer *n* and readings past the cable —
  `read_res2dinv(path, n_electrodes=N)` in Python or
  `shallowgeo ert-clean path -e N` in a terminal. Check that the break in the
  data agrees with the number of electrodes in the field notes.
- **Check the data with tests that need no inversion**: how fast ρa can
  change with *n*, whether readings from absent electrodes look different
  from real ones, and whether readings follow the geometry or the receiver
  channel.
- **An inversion always returns a picture.** The misfit (χ², RMS and its
  pattern), the coverage and the DOI index say which parts of it the data
  support; trying several reasonable settings says which features are robust.